In [1]:
from pmo_func import retriver,reranker,Classifier,summarizer
from dotenv import load_dotenv
import os
import requests
import requests
from bs4 import BeautifulSoup
import time
from typing import Dict, List, Any, Optional
import pandas as pd
import trafilatura as tra
import torch
from transformers import AutoModel,AutoTokenizer,AutoModelForCausalLM

e:\GENAI H2S\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:

class FactChecker:
    def __init__(self):
        self.factcheck_api = "https://factchecktools.googleapis.com/v1alpha1/claims:search"
        self.google_search = "https://www.google.com/search"
        self.reranker = None
        self.classifier = None
        self.summarizer = None

    def check_google_factcheck(self, claim: str , pages:int=5):
    
        load_dotenv()
        api_key = os.getenv("GOOGLE_FACT_CHECK_API")
        
        if not api_key:
            print("Google FactCheck API key not found")
            return None
        
        params = {
            'key': api_key,
            'query': claim,
            'languageCode': 'en-US',
            'pageSize': pages
        }
        
        try:
            response = requests.get(self.factcheck_api, params=params)
            response.raise_for_status()
            data = response.json()
            
            if 'claims' in data and data['claims']:
                # Return the most relevant fact-check
                claim_data = data['claims'][0]
                review = claim_data.get('claimReview', [{}])[0]
                
                return {
                    'claim': claim_data.get('text', ''),
                    'verdict': review.get('textualRating', 'Unknown'),
                    'summary': f"Rated {review.get('textualRating', 'Unknown')} by {review.get('publisher', {}).get('name', 'Unknown')}",
                    'source': review.get('url', ''),
                    'confidence': 'high',  # From official fact-checkers
                    'method': 'google_factcheck',
                    'URLs': [review.get('url', '')]
                }
            
        except Exception as e:
            print(f"FactCheck API error: {e}")
        
        return None
    
    def search_and_analyze_claim(self, claim: str):
        """
        Fallback method: Search web and analyze results with your models
        """
        print("No FactCheck result found, performing web analysis...")
        
        self.classifier = Classifier()
        self.summarizer = summarizer()
        self.reranker = reranker()


        top_evidences,urls,article_list = self.google_news_search(claim)
        
        if not top_evidences:
            return {
                'claim': claim,
                'verdict': 'Unverifiable',
                'summary': 'No relevant sources found to verify this claim',
                # 'confidence': 'low',
                'method': 'web_search',
                'soruce':"nothing",
                'URLs':""
            }
        
        # 2. Rerank articles by relevance
        reranked_articles = self.reranker.rerank_evidendce(claim,top_evidences)
        
        # 3. Classify articles stance
        verdict,_ = self.classifier(claim,reranked_articles)
        
        # 4. Generate summary verdict
        verdict,summary = self.summarizer(claim,top_evidences,verdict)
        
        return {
            'claim': claim,
            'verdict': verdict,
            'summary': summary,
            'source': [arc.get('source','') for arc in article_list],
            'method': 'web_analysis',
            'URLs':urls
        },article_list
    
    def google_news_search(self,query:str,num_pages:int = 1):
        print("Searching the Web")
        headers = {
            "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                        "AppleWebKit/537.36 (KHTML, like Gecko) "
                        "Chrome/115.0.0.0 Safari/537.36"
                        }
        
        articles_gg= []
        for pages in range(num_pages):
            params = {
                "q":query,
                "tbm":"nws",
                'start':int(pages) * 10 
                }

            try:
                res = requests.get(self.google_search,params=params,headers=headers,timeout=15)
                soup = BeautifulSoup(res.text,'html.parser')

                article_list=soup.select("div.SoaBEf a")
                if not article_list:
                    print("None Articles found")
                for article in article_list:
                    h1 = article.find('div',class_= "n0jPhd ynAwRc MBeuO nDgy9d").text
                    h2 = article.find('div',class_ = "GI74Re nDgy9d").text
                    title = h1 + h2

                    a_url = article['href']
                    time = article.find('div',class_="OSrXXb rbYSKb LfVVr").text
                    source  = article.find('div',class_ = "MgUUmf NUnG9d").text

                    try:
                        down = tra.fetch_url(a_url)
                        content = tra.extract(down) if down else "none extracted"
                        content = content if content else "No content extracted"
                    except Exception as e:
                        content = f"Error: {e}"

                    articles_gg.append({
                        "title":title,
                        'url':a_url,
                        'text':content,
                        'pblished_date':time,
                        'source':source
                    })
                
            except requests.exceptions.RequestException as e:
                print(f"Error Fething Google search | {e}")

            except Exception as e:
                print(f"Unforseen Error | {e}")

        top_evidences = [dict.get('text') for dict in articles_gg]
        urls = [dict.get('url') for dict in articles_gg]
        print("Web search Successfull")
        return top_evidences,urls,articles_gg
    
    def check_claim(self, claim: str):
        """
        Main function to check a claim using the complete pipeline.
        Always returns a tuple of (result_dict, article_list).
        """
        print(f"\n--- Checking claim: '{claim}' ---")
        
        factcheck_result = self.check_google_factcheck(claim)
        if factcheck_result:
            print("Found result in FactCheck database.")
            # FIXED: Return a tuple with None for the articles to keep the format consistent
            return factcheck_result, None
        
        print("No FactCheck result, falling back to web analysis...")
        # This already returns a tuple, so it's fine
        return self.search_and_analyze_claim(claim)
    
# Example usage
checker = FactChecker()
    
    # Test claims
test_claims = [
#         "COVID-19 vaccines contain microchips",
        # "The earth is flat",
        # "A new study shows chocolate prevents aging",  # This might not be in FactCheck DB
        "Nepal elected their New PM thorugh discord",
        # "is mumbai the capital of india",
        # "is  sunnny leone a pornstar",
        # "MS Dhoni does hookah party"
    ]
    
for claim in test_claims:
    result,arc = checker.check_claim(claim)
    print(f"\nClaim: {claim}")
    print(f"Verdict: {result['verdict']}")
    print(f"Explanation: {result['summary']}")
    print(f"Method: {result['method']}")
    print(f"source:{result['source']}")
    print(f"url:{result['URLs']}")
    # print(f"Confidence: {result['confidence']}")f"
    print("-" * 50)
    arc


--- Checking claim: 'Nepal elected their New PM thorugh discord' ---
No FactCheck result, falling back to web analysis...
No FactCheck result found, performing web analysis...
Classifier device: cuda
Summarizer device: cuda


You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


Got the reranker
Searching the Web
Web search Successfull

Claim: Nepal elected their New PM thorugh discord
Verdict: TRUE
Explanation: Kathmandu, Nepal – As Nepal burned on Thursday after two days of deadly unrest that ousted a government accused of corruption, thousands of young people gathered in a heated debate to decide their nation’s next leader
Method: web_analysis
source:['Al Jazeera', 'The Indian Express', 'The New York Times', 'The Times of India', 'DD News', 'Hindustan Times', 'Mint', 'The Logical Indian', 'The Independent', 'India Today']
url:['https://www.aljazeera.com/news/2025/9/15/more-egalitarian-how-nepals-gen-z-used-gaming-app-discord-to-pick-pm', 'https://indianexpress.com/article/explained/explained-global/nepal-pm-discord-gaming-chat-app-gen-z-protests-10248009/', 'https://www.nytimes.com/2025/09/11/world/asia/nepal-protest-genz-discord.html', 'https://timesofindia.indiatimes.com/technology/tech-news/nepals-gen-z-chooses-discord-to-elect-interim-pm-what-is-this-ap

In [31]:
torch.cuda.empty_cache()

In [4]:
arc

[{'title': 'Phones off, no online transactions: Khedkar family vanishes as Navi Mumbai, Pune police launch manhuntAn official said that they suspect the accused will soon apply for anticipatory bail and may try to remain on the run till the time they...',
  'url': 'https://indianexpress.com/article/cities/mumbai/phones-off-no-online-transactions-khedkar-family-vanishes-as-navi-mumbai-pune-police-launch-manhunt-10254893/',
  'text': 'Three days after rescuing a truck cleaner from sacked IAS officer Puja Khedkar’s Pune residence, the Navi Mumbai police are still on the lookout for her family members, who have allegedly switched off their mobile phones. The family members have also not made any online transactions since Sunday, an official said.\n“We are trying to trace the family, but they have switched off all their devices and are off the grid. We are now looking at CCTV cameras around the bungalow to trace what vehicle was used to escape,” a police official said.\nWhile the Navi Mumba

In [ ]:
class BetterSummarizer:
    def __init__(self):
        # Use a model better suited for reasoning and explanation
        self.model_name = "microsoft/DialoGPT-large"  # Or "microsoft/DialoGPT-medium" for less memory usage
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        print("Better Summarizer device:", self.device)
        
        try:
            self.tokenizer = AutoTokenizer.from_pretrained(self.model_name)
            self.model = AutoModelForCausalLM.from_pretrained(
                self.model_name,
                dtype=torch.float16 if self.device.type == "cuda" else torch.float32,
                low_cpu_mem_usage=True
            )
            self.model.to(self.device)
            self.model.eval()
        except Exception as e:
            print(f"Could not load better summarizer: {e}")
            # Fall back to the original summarizer
            self.fallback = summarizer()
    
    def forward(self, claim, top_evidence, verdict):
        # Extract just the text from the (score, text) tuples
        evidence_texts = [evidence[1] for evidence in top_evidence[:3]]  # Use only top 3 evidences
        
        if not evidence_texts:
            raise ValueError("No evidence provided")
        
        # Create a concise summary of the evidence
        concise_evidence = []
        for i, evidence in enumerate(evidence_texts):
            # Extract the most relevant sentences (first few sentences)
            sentences = evidence.split('. ')
            concise_evidence.append(f"Source {i+1}: {' '.join(sentences[:2])}.")
        
        evidence_summary = "\n".join(concise_evidence)
        
        # Create a more conversational prompt
        prompt = f"""
        As a fact-checker, I need to verify this claim: "{claim}"
        
        Here's what I found from reliable sources:
        {evidence_summary}
        
        Based on this evidence, my initial assessment is that the claim is {verdict}.
        
        Please provide a clear, concise explanation (2-3 sentences) of why this verdict is appropriate, 
        citing specific evidence from the sources above.
        """
        
        try:
            inputs = self.tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024).to(self.device)
            
            with torch.no_grad():
                outputs = self.model.generate(
                    inputs.input_ids,
                    max_length=300,
                    num_return_sequences=1,
                    temperature=0.7,
                    do_sample=True,
                    pad_token_id=self.tokenizer.eos_token_id
                )
            
            explanation = self.tokenizer.decode(outputs[0], skip_special_tokens=True)
            # Remove the prompt from the response
            explanation = explanation.replace(prompt, "").strip()
            
            return verdict, explanation
        except Exception as e:
            print(f"Error with better summarizer: {e}")
            # Fall back to the original summarizer
            if hasattr(self, 'fallback'):
                return self.fallback(claim, top_evidence, verdict)
            return verdict, "Could not generate explanation"
    
    def __call__(self, claim, top_evidence, verdict):
        return self.forward(claim, top_evidence, verdict)